# Notebook 03 — Funciones definidas por el usuario

Tercer sub-bloque del Tema 06. Con bloques anónimos y control de flujo (Notebooks 01-02) ya puedes ejecutar lógica procedural, pero solo ad-hoc. Ahora empaquetas esa lógica en **funciones** que viven en la base, se pueden invocar desde queries (`SELECT mi_funcion(...)`), e incluso indexar.

Vas a ver: la sintaxis `CREATE FUNCTION`, los tipos de retorno (escalar, `SETOF`, `TABLE`), `LANGUAGE plpgsql` vs `LANGUAGE sql`, las **categorías de volatilidad** (`IMMUTABLE`/`STABLE`/`VOLATILE`) y cuándo cada una. Cerramos con casos prácticos sobre Northwind DWH.

**Contenido de este notebook:**

- [Setup](#setup)
- [Tu primera función — `CREATE FUNCTION`](#tu-primera-función--create-function)
- [Parámetros y tipos de retorno escalares](#parámetros-y-tipos-de-retorno-escalares)
- [`LANGUAGE plpgsql` vs `LANGUAGE sql`](#language-plpgsql-vs-language-sql)
- [Funciones que devuelven `SETOF` y `TABLE`](#funciones-que-devuelven-setof-y-table)
- [Categorías de volatilidad](#categorías-de-volatilidad)
- [`OR REPLACE`, `DROP FUNCTION` y firmas](#or-replace-drop-function-y-firmas)
- [Caso práctico sobre Northwind DWH](#caso-práctico-sobre-northwind-dwh)

## Setup

In [ ]:
# Setup — instala JupySQL si hace falta (Colab trae ipython-sql, no JupySQL).
import importlib.util, subprocess, sys
if importlib.util.find_spec("jupysql") is None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "ipython-sql"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupysql"], check=True)
    print("⚠ JupySQL instalado. REINICIA el kernel (Entorno de ejecución → Reiniciar sesión)")
    print("  y vuelve a correr esta celda y las siguientes.")
else:
    print("✓ JupySQL listo.")

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

# Reemplaza con tus valores del Tema 01
AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine
# Devuelve cada query como DataFrame de pandas (mejor render en Colab, y
# el resultado es directamente manipulable con pandas).
%config SqlMagic.autopandas = True

## Tu primera función — `CREATE FUNCTION`

Sintaxis básica:

```sql
CREATE FUNCTION nombre(parametros) RETURNS tipo AS $$
DECLARE
    ...
BEGIN
    ...
    RETURN valor;
END;
$$ LANGUAGE plpgsql;
```

Comparada con un bloque anónimo: la función **se guarda** en la base con un nombre, **acepta parámetros**, y **devuelve un valor**. Después la invocas como cualquier función built-in.

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION saludar(nombre TEXT) RETURNS TEXT AS $$
BEGIN
    RETURN 'Hola, ' || nombre || '!';
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
SELECT saludar('Mundo')        AS saludo_1,
       saludar('Diplomado')    AS saludo_2;

Una vez creada, la función queda como **objeto de primera clase** en tu schema. La puedes usar en `SELECT`, dentro de un `WHERE`, dentro de un `JOIN`, etc. — donde sea que se puedan usar funciones built-in.

**`CREATE OR REPLACE`** te permite redefinir la función si ya existe — útil mientras desarrollas. Sin `OR REPLACE`, redefinir falla si la función ya existe.

## Parámetros y tipos de retorno escalares

Los parámetros tienen sintaxis `nombre TIPO` y se separan por coma:

```sql
CREATE FUNCTION calcular_descuento(
    precio    NUMERIC,
    porcentaje NUMERIC
) RETURNS NUMERIC AS $$
BEGIN
    RETURN precio * (1 - porcentaje / 100);
END;
$$ LANGUAGE plpgsql;
```

Tipos de retorno escalares comunes: `NUMERIC`, `INTEGER`, `TEXT`, `BOOLEAN`, `DATE`, `TIMESTAMP`.

**Parámetros con default:**

```sql
CREATE FUNCTION calcular_descuento(
    precio    NUMERIC,
    porcentaje NUMERIC DEFAULT 10
) RETURNS NUMERIC AS $$ ... $$ LANGUAGE plpgsql;

-- Después puedes llamarla con uno o dos argumentos:
SELECT calcular_descuento(100);          -- usa porcentaje = 10 por default
SELECT calcular_descuento(100, 25);      -- override
```

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION precio_con_descuento(
    precio     NUMERIC,
    porcentaje NUMERIC DEFAULT 10
) RETURNS NUMERIC AS $$
BEGIN
    IF porcentaje < 0 OR porcentaje > 100 THEN
        RAISE EXCEPTION 'Porcentaje fuera de rango: %', porcentaje;
    END IF;
    
    RETURN ROUND(precio * (1 - porcentaje / 100), 2);
END;
$$ LANGUAGE plpgsql;

In [ ]:
%%sql
SELECT precio_con_descuento(100)         AS sin_arg,
       precio_con_descuento(100, 25)     AS con_25;

In [ ]:
%%sql
-- Usar la función directamente en un SELECT sobre fact_sales
SELECT
    order_id,
    unit_price,
    precio_con_descuento(unit_price, 15) AS con_promocion_15
FROM northwind_dwh.fact_sales
LIMIT 5;

## `LANGUAGE plpgsql` vs `LANGUAGE sql`

PostgreSQL soporta varios lenguajes para escribir funciones. Los dos más usados son:

- **`LANGUAGE plpgsql`** — el procedural (`IF`, `LOOP`, `DECLARE`, etc.).
- **`LANGUAGE sql`** — funciones definidas como una sola query SQL (sin lógica procedural).

Comparación:

```sql
-- Versión SQL: una sola query, sin BEGIN/END
CREATE FUNCTION duplicar(n INTEGER) RETURNS INTEGER
AS 'SELECT n * 2'
LANGUAGE sql IMMUTABLE;

-- Versión PL/pgSQL: bloque procedural
CREATE FUNCTION duplicar(n INTEGER) RETURNS INTEGER AS $$
BEGIN
    RETURN n * 2;
END;
$$ LANGUAGE plpgsql IMMUTABLE;
```

**Regla:** si tu lógica cabe en una sola query SQL, **usa `LANGUAGE sql`** — el optimizador puede *inlinear* la función directamente en la query que la llama, lo que es mucho más rápido. Para lógica procedural real (con loops, condicionales complejos), `LANGUAGE plpgsql`.

In [ ]:
%%sql
-- Versión SQL pura — más simple y más rápida cuando aplica
CREATE OR REPLACE FUNCTION duplicar_sql(n INTEGER) RETURNS INTEGER
AS 'SELECT n * 2'
LANGUAGE sql IMMUTABLE;

SELECT duplicar_sql(5) AS resultado;

## Funciones que devuelven `SETOF` y `TABLE`

Hasta ahora las funciones devuelven **un solo valor escalar**. PostgreSQL también permite que una función devuelva **un conjunto de filas** — se comporta como una tabla en el `FROM`.

### `RETURNS SETOF tipo`

Cada fila es un valor del tipo declarado. Útil cuando devuelves varias instancias de un tipo simple:

```sql
CREATE FUNCTION pares_hasta(n INTEGER) RETURNS SETOF INTEGER AS $$
BEGIN
    FOR i IN 0..n BY 2 LOOP
        RETURN NEXT i;
    END LOOP;
END;
$$ LANGUAGE plpgsql IMMUTABLE;

SELECT * FROM pares_hasta(10);
```

**`RETURN NEXT`** agrega una fila al resultado pero **no termina** la función — sigue ejecutando hasta el `END` o un `RETURN` sin valor.

### `RETURNS TABLE(...)`

Devuelve filas con **varias columnas**, cada una con su tipo declarado. Más útil para funciones que parecen "una query reutilizable":

```sql
CREATE FUNCTION ventas_por_categoria(anio INTEGER)
RETURNS TABLE(categoria TEXT, total NUMERIC) AS $$
BEGIN
    RETURN QUERY
    SELECT dp.category_name, SUM(fs.line_total)
      FROM northwind_dwh.fact_sales fs
      JOIN northwind_dwh.dim_product dp USING (product_key)
      JOIN northwind_dwh.dim_date    dd ON dd.date_key = fs.order_date_key
     WHERE dd.year = anio
     GROUP BY dp.category_name;
END;
$$ LANGUAGE plpgsql STABLE;
```

**`RETURN QUERY`** ejecuta una query y agrega todas sus filas al resultado de la función.

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION pares_hasta(n INTEGER) RETURNS SETOF INTEGER AS $$
BEGIN
    FOR i IN 0..n BY 2 LOOP
        RETURN NEXT i;
    END LOOP;
END;
$$ LANGUAGE plpgsql IMMUTABLE;

In [ ]:
%%sql
-- Se invoca en el FROM como si fuera tabla
SELECT * FROM pares_hasta(10);

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION ventas_por_categoria(anio INTEGER)
RETURNS TABLE(categoria TEXT, total NUMERIC) AS $$
BEGIN
    RETURN QUERY
    SELECT dp.category_name::TEXT, ROUND(SUM(fs.line_total), 2)
      FROM northwind_dwh.fact_sales fs
      JOIN northwind_dwh.dim_product dp USING (product_key)
      JOIN northwind_dwh.dim_date    dd ON dd.date_key = fs.order_date_key
     WHERE dd.year = anio
     GROUP BY dp.category_name
     ORDER BY 2 DESC;
END;
$$ LANGUAGE plpgsql STABLE;

In [ ]:
%%sql
SELECT * FROM ventas_por_categoria(1997);

## Categorías de volatilidad

Cada función PostgreSQL se cataloga en una de tres categorías que le dicen al **optimizador** cuándo puede cachear su resultado. La declaración va al final de la definición:

| Categoría | Significado | Optimizaciones que habilita |
|---|---|---|
| **`IMMUTABLE`** | Mismo input → mismo output, **siempre**. No toca la base. | Cachear resultado, usar en índices, evaluar en compile-time |
| **`STABLE`** | Mismo input → mismo output **dentro de la misma query**. Lee de la base pero no escribe. | Reusar dentro de la misma query, NO indexable |
| **`VOLATILE`** (default) | El resultado puede cambiar entre llamadas — sin garantía. | Ninguna |

**Cómo decidir:**

- ¿Es matemática pura sin tocar la base? → `IMMUTABLE` (`pi()`, `redondear`, `formatear_fecha`).
- ¿Hace `SELECT` pero no `INSERT/UPDATE/DELETE`? → `STABLE` (`obtener_total_cliente`).
- ¿Modifica datos o usa `random()`/`now()` etc.? → `VOLATILE` (el default).

**Por qué importa:** una función `IMMUTABLE` puede usarse en un índice. Una `VOLATILE` no. Si declaras `STABLE` algo que es realmente `VOLATILE`, puedes obtener resultados raros al cachearse mal.

**Caveat de honestidad:** si declaras tu función como más "pura" de lo que es (e.g., `IMMUTABLE` cuando realmente lee de la base), PostgreSQL te cree y optimiza basándose en eso. Resultados incorrectos no son bug del motor — son consecuencia de tu mentira.

## `OR REPLACE`, `DROP FUNCTION` y firmas

**Identidad de funciones:** PostgreSQL identifica funciones por **nombre + tipos de los parámetros**, NO solo por nombre. Es lo que se llama "firma" (*signature*). Puedes tener dos funciones con el mismo nombre pero distintos parámetros (overloading):

```sql
CREATE FUNCTION duplicar(n INTEGER) RETURNS INTEGER ...
CREATE FUNCTION duplicar(n NUMERIC) RETURNS NUMERIC ...
-- Son funciones distintas. PostgreSQL elige según el tipo del argumento.
```

**Para eliminar:** debes especificar la firma:

```sql
DROP FUNCTION duplicar(INTEGER);     -- elimina solo la versión INTEGER
DROP FUNCTION duplicar(NUMERIC);     -- elimina solo la versión NUMERIC
```

**`IF EXISTS`** evita error si la función no existe (útil para scripts idempotentes):

```sql
DROP FUNCTION IF EXISTS duplicar(INTEGER);
```

In [ ]:
%%sql
-- Limpieza: eliminar las funciones que creamos en este notebook
DROP FUNCTION IF EXISTS saludar(TEXT);
DROP FUNCTION IF EXISTS precio_con_descuento(NUMERIC, NUMERIC);
DROP FUNCTION IF EXISTS duplicar_sql(INTEGER);
DROP FUNCTION IF EXISTS pares_hasta(INTEGER);
DROP FUNCTION IF EXISTS ventas_por_categoria(INTEGER);

## Caso práctico sobre Northwind DWH

Función que devuelve el **resumen de ventas de un cliente**: total, número de pedidos, promedio por pedido. Mostrar uso real en un reporte.

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION resumen_cliente(customer_natural_key TEXT)
RETURNS TABLE(
    customer    TEXT,
    pedidos     BIGINT,
    ventas      NUMERIC,
    promedio    NUMERIC
) AS $$
BEGIN
    RETURN QUERY
    SELECT
        dc.company_name::TEXT,
        COUNT(DISTINCT fs.order_id),
        ROUND(SUM(fs.line_total), 2),
        ROUND(SUM(fs.line_total) / NULLIF(COUNT(DISTINCT fs.order_id), 0), 2)
    FROM northwind_dwh.fact_sales fs
    JOIN northwind_dwh.dim_customer dc USING (customer_key)
    WHERE dc.customer_id = customer_natural_key
    GROUP BY dc.company_name;
END;
$$ LANGUAGE plpgsql STABLE;

In [ ]:
%%sql
SELECT * FROM resumen_cliente('ALFKI');

In [ ]:
%%sql
-- Función IMMUTABLE para clasificar montos en bandas
CREATE OR REPLACE FUNCTION banda_venta(monto NUMERIC) RETURNS TEXT
AS $$
    SELECT CASE
        WHEN monto < 100   THEN 'micro'
        WHEN monto < 500   THEN 'pequeña'
        WHEN monto < 2000  THEN 'mediana'
        ELSE 'grande'
    END
$$ LANGUAGE sql IMMUTABLE;

-- Uso en una query analítica
SELECT banda_venta(line_total) AS banda,
       COUNT(*)                AS lineas,
       SUM(line_total)         AS total
FROM   northwind_dwh.fact_sales
GROUP BY banda_venta(line_total)
ORDER BY 3 DESC;

In [ ]:
%%sql
-- Limpieza
DROP FUNCTION IF EXISTS resumen_cliente(TEXT);
DROP FUNCTION IF EXISTS banda_venta(NUMERIC);

## Cierre

Lo que cubriste:

| Tema | Construcción clave |
|---|---|
| Crear función | `CREATE [OR REPLACE] FUNCTION nombre(params) RETURNS tipo AS $$ ... $$ LANGUAGE plpgsql;` |
| Parámetros con default | `param TIPO DEFAULT valor` |
| SQL vs plpgsql | `LANGUAGE sql` para una query, `plpgsql` para procedural real |
| Retorno escalar | `RETURNS NUMERIC` (etc.) + `RETURN valor;` |
| Retorno set simple | `RETURNS SETOF tipo` + `RETURN NEXT valor;` |
| Retorno set complejo | `RETURNS TABLE(col1 t1, col2 t2)` + `RETURN QUERY SELECT...;` |
| Volatilidad | `IMMUTABLE` (pura) / `STABLE` (lee) / `VOLATILE` (default) |
| Eliminar | `DROP FUNCTION nombre(tipo_params)` — la firma es nombre + tipos |

El siguiente notebook (**04 — Procedimientos y cursores**) introduce `CREATE PROCEDURE` — el primo de las funciones que sí puede controlar transacciones y se invoca con `CALL` en lugar de `SELECT`. También cubrimos cursores explícitos, que aunque rara vez son la herramienta correcta, valen la pena conocer para reconocerlos en código heredado.

---

<p align="center">
<a href="02_control_de_flujo.ipynb">← Anterior: Notebook 02</a> | <a href="Readme.md">Volver al índice</a> | <a href="04_procedimientos_y_cursores.ipynb">Siguiente: Notebook 04 — Procedimientos y cursores →</a>
</p>